# 16 - Deploy the reviewed Azure platform

Run only after 15 passes and you have reviewed its Create/NoChange list. This notebook creates the eight service/storage resources and three scoped workspace identity roles in infra/platform.json. Basic ACR starts its daily charge; storage, logs, and other services can charge for usage. The 91.90 THB plan is an estimate, not a bill or cap. No credits are assumed. The 48-hour cleanup tag is a reminder, not automatic deletion.

There is no training VM, running prediction app, Dedicated profile, private endpoint, or delete command in this step. Deployment uses Incremental mode and checks the same subscription, group ownership, preview age, and file hashes.


In [4]:
from pathlib import Path
import json
import platform
import subprocess
import sys

assert platform.system() == "Linux", "Use the WSL kernel."
assert Path(sys.prefix).name == "mlop-pj", "Select the project environment."
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if p.name == "MLOP-PJ" and (p / "README.md").is_file())

preview = json.loads((ROOT / "reports/azure-platform-preview.json").read_text())
assert preview["preview_ready"] and not preview["errors"], "Finish 15 first."
print("Group:", preview["resource_group"])
print("Cleanup by UTC:", preview["cleanup_by_utc"])
print("Review these changes before running the next cell:")
print(json.dumps(preview["changes"], indent=2))


Group: rg-mlop-pj-492ea24d
Cleanup by UTC: 2026-10-13T14:15:40.612068+00:00
Review these changes before running the next cell:
[
  {
    "change": "Create",
    "resource": "Microsoft.ContainerRegistry/registries/acrmloppj492ea24d/providers/Microsoft.Authorization/roleAssignments/1f2ee09a-5a78-5863-818d-5cb24a752c03"
  },
  {
    "change": "Create",
    "resource": "Microsoft.KeyVault/vaults/kv-mlop-pj-492ea24d"
  },
  {
    "change": "Create",
    "resource": "Microsoft.KeyVault/vaults/kv-mlop-pj-492ea24d/providers/Microsoft.Authorization/roleAssignments/2adc23dc-cece-54be-82ec-a6b989ff3977"
  },
  {
    "change": "Create",
    "resource": "Microsoft.MachineLearningServices/workspaces/ml-mlop-pj-492ea24d"
  },
  {
    "change": "Create",
    "resource": "Microsoft.Storage/storageAccounts/stmloppj492ea24d/providers/Microsoft.Authorization/roleAssignments/20e8503f-2538-5669-a115-02e71b00484c"
  }
]


## Create the platform

This cell starts applicable Azure charges. Run once after reviewing the preview. Provisioning may take several minutes. Expect errors: [], deployment_state: Succeeded, and platform_created: true. If it fails or times out, save the result and stop; paid resources may already exist or deployment may still be running. Do not rerun blindly.


In [5]:
result = subprocess.run([sys.executable, "-m", "cloudlayer.platform_deploy"],
                        cwd=ROOT, capture_output=True, text=True)
if result.returncode:
    print(result.stderr)
    raise RuntimeError("Deployment script failed. Read the traceback above.")
report = json.loads(result.stdout)
print(json.dumps(report, indent=2))
assert not report["errors"] and report["platform_created"], "Stop and send the deployment report."


{
  "checked_at": "2026-10-11T14:57:18.003368+00:00",
  "apply_started": true,
  "paid_resources_may_exist": true,
  "platform_created": false,
  "errors": [
    "RuntimeError: Azure CLI failed: DeploymentFailed, RoleAssignmentExists"
  ],
  "resource_group": "rg-mlop-pj-492ea24d",
  "region": "centralindia",
  "suffix": "492ea24d",
  "project_tag": "mlop-pj-6688012",
  "budget_thb_before_credits": 150,
  "cleanup_by_utc": "2026-10-13T14:15:40.612068+00:00",
  "template_file": "infra/platform-recovery.json",
  "next": "Deployment may be partial or still running. Check the project group before retrying; do not assume zero cost."
}


AssertionError: Stop and send the deployment report.

## Output

Save this notebook. Output: reports/azure-platform-deploy.json. Send results from 15 and 16. Next: check environment CPU quota and identity/data access, push the tested image, and connect cloud training and monitoring. Platform creation does not prove the API or pipeline works.
